In [3]:
!pip install twilio flask pyngrok transformers torch scipy pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 87.4 MB/s eta 0:00:00


In [4]:
# ==========================================
# STEP 1: INSTALL DEPENDENCIES (Run this first!)
# ==========================================
# We explicitly install 'twilio' along with other libraries
!pip install twilio flask pyngrok transformers torch scipy pillow

# ==========================================
# STEP 2: LOAD AI MODELS
# ==========================================
import torch
from transformers import pipeline
from PIL import Image
import io

print("⏳ Loading AI Models... (This may take a minute)")

# 1. RISK ENGINE (Zero-Shot Classification)
risk_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
risk_labels = ["Critical Emergency", "Severe Side Effect", "Moderate Concern", "General Query", "Safe/Positive Feedback"]

# 2. ENTITY EXTRACTOR (NER)
ner_pipeline = pipeline("ner", model="d4data/biomedical-ner-all", aggregation_strategy="simple")

# 3. PHARMACIST ASSISTANT (QA)
qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2")

print("✅ All AI Models Loaded Successfully!")

# ==========================================
# STEP 3: DEFINE INTELLIGENCE FUNCTIONS
# ==========================================

def get_risk_level(text):
    result = risk_classifier(text, risk_labels)
    top_label = result['labels'][0]
    
    if top_label == "Critical Emergency":
        return 5, "URGENT: Emergency Protocol Initiated"
    elif top_label == "Severe Side Effect":
        return 4, "High Risk: Doctor Review Required"
    elif top_label == "Moderate Concern":
        return 3, "Moderate: Queued for Review"
    elif top_label == "General Query":
        return 2, "Low Risk: Automated Guidance"
    else:
        return 1, "Safe: Feedback Recorded"

def analyze_medical_entities(text):
    entities = ner_pipeline(text)
    clean_entities = []
    for ent in entities:
        if ent['score'] > 0.8:
            clean_entities.append(f"{ent['word']} ({ent['entity_group']})")
    return ", ".join(clean_entities)

def helper_bot_answer(question):
    sop_context = """
    Paracetamol dosage is 500mg every 4-6 hours. Do not exceed 4000mg in 24 hours.
    Amoxicillin is an antibiotic used for bacterial infections. Take with food.
    Level 5 risks include difficulty breathing, anaphylaxis, and severe chest pain.
    Store insulin in the refrigerator between 2 to 8 degrees Celsius.
    """
    result = qa_pipeline(question=question, context=sop_context)
    return result['answer']

# ==========================================
# STEP 4: THE FLASK SERVER
# ==========================================
from flask import Flask, request
from pyngrok import ngrok
from twilio.twiml.messaging_response import MessagingResponse # IMPORT FIXED HERE

app = Flask(__name__)

@app.route("/bot", methods=['POST'])
def bot():
    incoming_msg = request.values.get('Body', '').lower()
    num_media = int(request.values.get('NumMedia', 0))
    sender = request.values.get('From', '')

    print(f"📩 Received from {sender}: {incoming_msg}")

    response_text = ""

    if num_media > 0:
        response_text = "📷 Image received. Analyzing for visual symptoms... [Vision AI Placeholder Active]"
    elif incoming_msg.startswith("question"):
        answer = helper_bot_answer(incoming_msg)
        response_text = f"🤖 Bot: {answer}"
    else:
        level, message = get_risk_level(incoming_msg)
        entities = analyze_medical_entities(incoming_msg)

        response_text = f"*Medicova AI Analysis* 🛡️\n\n"
        response_text += f"⚠️ *Risk Assessment:* Level {level}\n"
        response_text += f"📋 *Status:* {message}\n"
        if entities:
            response_text += f"🔍 *Detected:* {entities}\n"

        if level >= 4:
            response_text += "\n🚨 *ESCALATION:* A doctor has been alerted immediately."

    resp = MessagingResponse()
    resp.message(response_text)
    return str(resp)

# ==========================================
# STEP 5: RUN THE SERVER
# ==========================================

# IMPORTANT: PASTE YOUR NGROK TOKEN BELOW
NGROK_AUTH_TOKEN = "38w5AbRYF3T9OYapHg2O3TWIbWR_4rt5tbGDoxtgeNvxZYSZT" 

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
# Terminate open tunnels to avoid errors
ngrok.kill()
public_url = ngrok.connect(5000).public_url

print(f"🚀 YOUR SERVER IS LIVE! COPY THIS URL INTO TWILIO: {public_url}/bot")

app.run(port=5000)

⏳ Loading AI Models... (This may take a minute)


Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0


✅ All AI Models Loaded Successfully!
🚀 YOUR SERVER IS LIVE! COPY THIS URL INTO TWILIO: https://bionic-nonelaborate-tandy.ngrok-free.dev/bot
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [29/Jan/2026 15:01:23] "POST /bot HTTP/1.1" 200 -


📩 Received from whatsapp:+919428416735: i have a severe headache and difficulty breathing.
